## Install Required Dependencies for IO

Installs the `fastparquet` library to enable reading and writing Parquet files, a fast, efficient data storage format. This is needed for saving/loading large tabular data such as merged YouTube datasets.

In [ ]:
# !pip install pyarrow
!pip install fastparquet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fastparquet]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


## Merge Video Metadata and Transcript Datasets

- Loads cleaned YouTube metadata and transcript files as DataFrames.
- Handles any format or delimiter issues in the transcript CSV.
- Merges the two DataFrames on the video ID (`id`), keeping only videos with an available transcript.
- Drops any records with empty transcript fields.
- Saves the merged data as both CSV and Parquet for downstream processing.

In [9]:
import pandas as pd

# Load datasets
meta_df = pd.read_csv("../data/flagged dataset/Master_Task1_withTranscriptFlag.csv")

# Try reading with a different engine and error handling
try:
    trans_df = pd.read_csv("../data/processed/Master_task2_Cleaned_main.csv", engine='python', on_bad_lines='skip')
except Exception as e:
    print(f"Error reading CSV: {e}")
    # If still fails, try reading with a different delimiter or quoting
    trans_df = pd.read_csv("../data/processed/Master_task2_Cleaned_main.csv", engine='python', sep='\t', on_bad_lines='skip')


# Ensure consistent column name for merging
trans_df.rename(columns={"video_id": "id"}, inplace=True)

# Merge on video ID
merged_df = pd.merge(meta_df, trans_df, on="id", how="inner")

# Remove empty or missing transcripts
merged_df["transcript"] = merged_df["transcript"].fillna("").astype(str)
merged_df = merged_df[merged_df["transcript"].str.strip() != ""]

print(f"✅ Merged dataset shape: {merged_df.shape}")

# Save outputs
merged_df.to_csv("../data/Embeddings/Merged_VideoData.csv", index=False)
merged_df.to_parquet("../data/Embeddings/Merged_VideoData.parquet", index=False, engine="fastparquet")


print("💾 Saved merged dataset as CSV and Parquet.")

✅ Merged dataset shape: (607, 25)
💾 Saved merged dataset as CSV and Parquet.


## Install and Upgrade Embedding Libraries

Installs (or upgrades) the `sentence-transformers`, `torch`, `torchvision`, `torchaudio`, `transformers`, and `numpy` libraries. These are required for generating and processing text embeddings using state-of-the-art deep learning models.

In [ ]:
# !pip install sentence-transformers
# !pip install numpy

!pip install --force-reinstall --upgrade sentence-transformers torch torchvision torchaudio transformers numpy


  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached numpy-2.3.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached huggingface_hub-1.0.1-py3-none-any.whl.metadata (13 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2025.10.23-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
Using cached sentence_transfo

## Compute Text Embeddings for Videos

- Loads the merged video and transcript data.
- Initializes a pretrained SentenceTransformer model (`all-MiniLM-L6-v2`).
- Combines each video's title and transcript into a single string for embedding.
- Generates normalized vector embeddings for each combined text.
- Saves the resulting DataFrame, now including the new `embedding` column, as both CSV and Parquet files for efficient downstream use.

In [11]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load merged dataset
merged_df = pd.read_csv("../data/Embeddings/Merged_Embeddings.csv")

# Initialize model
model = SentenceTransformer("all-MiniLM-L6-v2")
# Prepare text for embeddings -faiz edits
merged_df = merged_df.loc[:, ~merged_df.columns.str.contains("^Unnamed")]

merged_df["title"] = merged_df["title"].fillna("").astype(str)
merged_df["transcript"] = merged_df["transcript"].fillna("").astype(str)
merged_df["combined_text"] = merged_df["title"] + " " + merged_df["transcript"]
merged_df = merged_df[merged_df["combined_text"].str.strip() != ""]

# Combine title and transcript for embeddings
# merged_df["combined_text"] = merged_df["title"] + " " + merged_df["transcript"]

# Generate embeddings
# embeddings = model.encode(merged_df["combined_text"].tolist(), show_progress_bar=True)
embeddings = model.encode(
    merged_df["combined_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


# Add embeddings to DataFrame
merged_df["embedding"] = embeddings.tolist()

# Save outputs
merged_df.to_csv("../data/Embeddings/Merged_Embeddings.csv", index=False)
merged_df.to_parquet("../data/Embeddings/Merged_Embeddings.parquet", index=False)

print(f"✅ Embeddings generated and saved for {len(merged_df)} videos.")


Batches: 100%|██████████| 20/20 [00:51<00:00,  2.58s/it]


✅ Embeddings generated and saved for 617 videos.


## Install ChromaDB for Persistent Vector Storage

Installs the `chromadb` library—required for efficiently storing and querying video embeddings with persistent database support.

In [12]:
!pip install chromadb

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 28.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 33.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 47.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 32.5 MB/s  0:00:00
  Created wheel for pypika: filename=pypika-0.48.9-py2.py3-none-any.whl size=53803 sha256=262c4f8c8fc627528f23bcc5b7e99b10572b14945b6894b4b5d76cfeb5811ab4
  Stored in directory: /home/codespace/.cache/pip/wheels/d5/3d/69/8d68d249cd3de2584f226e27fd431d6344f7d70fd856ebd01b
Successfully built pypika
  Attempting uninstall: urllib3━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  5/54 [uvloop]ets]
    Fou

## Store Embeddings in ChromaDB

- Loads the DataFrame of video embeddings.
- Connects to a persistent ChromaDB vector database.
- Ensures unique video IDs.
- Converts each embedding to a numeric NumPy array.
- Stores video embeddings, titles, and transcripts in the ChromaDB collection for subsequent semantic search.

In [13]:
import pandas as pd
import numpy as np
import chromadb

# ============================
# Load dataset from Parquet
# ============================
merged_df = pd.read_parquet("../data/Embeddings/Merged_Embeddings.parquet")

# Initialize persistent ChromaDB client
client = chromadb.PersistentClient(path="../chroma_db")

# Create or get collection
collection = client.get_or_create_collection(name="youtube_videos")

# Remove duplicate IDs
merged_df.drop_duplicates(subset=['id'], inplace=True)

# Convert embedding column safely (if stored as string)
def parse_embedding(x):
    if isinstance(x, str):
        return np.array(eval(x))
    return np.array(x)

merged_df["embedding"] = merged_df["embedding"].apply(parse_embedding)

# Stack embeddings into a single array
embeddings = np.vstack(merged_df["embedding"].values)

# Add data to ChromaDB
collection.add(
    ids=merged_df["id"].astype(str).tolist(),
    embeddings=embeddings,
    metadatas=merged_df[["title", "transcript"]].to_dict(orient="records"),
    documents=merged_df["combined_text"].astype(str).tolist()
)

print(f"✅ Stored {len(merged_df)} videos in ChromaDB collection 'youtube_videos'.")
print("🎯 Data is ready for semantic search queries.")


✅ Stored 541 videos in ChromaDB collection 'youtube_videos'.
🎯 Data is ready for semantic search queries.


## End-to-End Semantic Search Demonstration (Interactive)

Defines the interactive workflow for semantic search:

- Receives user search query via standard input.
- Generates the query embedding with SentenceTransformer.
- Searches the ChromaDB vector store for the most semantically similar videos.
- Formats and displays the top results, including video titles, preview of transcripts, and similarity scores.

This cell acts as a live test of the full search pipeline with your embedded data.

In [14]:
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

# ===============================
# 1️⃣ Query Input Handling
# ===============================
def get_user_query():
    query = input("🔍 Enter your search query: ").strip()
    if not query:
        raise ValueError("❌ Query cannot be empty. Please enter a valid search term.")
    # Optional preprocessing
    query = ''.join(c for c in query if c.isalnum() or c.isspace())
    return query


# ===============================
# 2️⃣ Query Embedding Generation
# ===============================
def generate_query_embedding(query):
    print("⚙️ Loading embedding model...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embedding = model.encode(query, convert_to_numpy=True)
    return embedding


# ===============================
# 3️⃣ Perform Semantic Search
# ===============================
def search_chromadb(query_embedding, top_k=5):
    print("🔎 Connecting to ChromaDB...")
    client = chromadb.PersistentClient(path="../chroma_db")
    collection = client.get_or_create_collection(name="youtube_videos")

    # Perform the semantic search
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["metadatas", "documents", "distances"]
    )
    return results


# ===============================
# 4️⃣ Format and Filter Results
# ===============================
def format_results(results, min_score=0.2):
    formatted = []
    for i in range(len(results["ids"][0])):
        score = 1 / (1 + results["distances"][0][i])
  # Convert distance to similarity
        if score >= min_score:
            data = {
                "rank": i + 1,
                "title": results["metadatas"][0][i].get("title", "N/A"),
                "transcript": results["metadatas"][0][i].get("transcript", "")[:200] + "...",
                "similarity_score": round(score, 3)
            }
            formatted.append(data)
    return formatted


# ===============================
# 5️⃣ Display Results
# ===============================
def display_results(formatted_results):
    if not formatted_results:
        print("❌ No relevant results found.")
        return
    print("\n🎯 Top Search Results:")
    for r in formatted_results:
        print(f"\nRank {r['rank']}")
        print(f"Title: {r['title']}")
        print(f"Similarity Score: {r['similarity_score']}")
        print(f"Transcript (Preview): {r['transcript']}")


# ===============================
# 🚀 Main Script
# ===============================
if __name__ == "__main__":
    try:
        query = get_user_query()
        query_embedding = generate_query_embedding(query)
        results = search_chromadb(query_embedding, top_k=5)
        formatted_results = format_results(results, min_score=0.2)
        display_results(formatted_results)
    except Exception as e:
        print(f"⚠️ Error: {e}")


⚙️ Loading embedding model...
🔎 Connecting to ChromaDB...

🎯 Top Search Results:

Rank 1
Title: javascript course for beginners
Similarity Score: 0.555
Transcript (Preview): learn javascript this course is designed to take beginners through the basics of javascript with clear explanations and quiz sections while we already have a lot of javascript courses on our channel s...

Rank 2
Title: 6 hours of javascript projects from beginner to advanced
Similarity Score: 0.453
Transcript (Preview): this video features over six hours of hands on javascript content. it's organized into 15 projects ranging from beginner to advanced, and you'll find the timestamps down below along with all of the li...

Rank 3
Title: html tutorial website crash course for beginners
Similarity Score: 0.446
Transcript (Preview): this is an html crash course. i'm beau carnes and i'm going to teach you the basics of html. let's jump right into it. you probably already know that html is used to create web pages. it sta

## Programmatic Semantic Search Function for Reuse

Implements functions to:

- Generate an embedding for any user-supplied query.
- Retrieve the top-k most similar video results from ChromaDB.
- Display results with titles, transcript previews, and similarity scores.

Can be integrated in a Python script or called interactively for repeated testing.

In [18]:
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

# ===============================
# 1️⃣ Generate query embedding
# ===============================
def generate_query_embedding(query_text):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    query_embedding = model.encode(query_text, convert_to_numpy=True)
    return query_embedding

# ===============================
# 2️⃣ Search top 5 results
# ===============================
def search_top_videos(query_text, top_k=5):
    # Load model & encode query
    embedding = generate_query_embedding(query_text)

    # Connect to ChromaDB
    client = chromadb.PersistentClient(path="../chroma_db")
    collection = client.get_or_create_collection(name="youtube_videos")

    # Perform semantic search
    results = collection.query(
        query_embeddings=embedding.tolist(),
        n_results=top_k,
        include=["metadatas", "documents", "distances"]
    )

    # Format results
    formatted_results = []
    for i in range(len(results["ids"][0])):
        score = 1 / (1 + results["distances"][0][i])
  # Convert distance to similarity
        formatted_results.append({
            "rank": i + 1,
            "title": results["metadatas"][0][i].get("title", "N/A"),
            "similarity_score": round(score, 3),
            "transcript": results["metadatas"][0][i].get("transcript", "")[:200] + "..."
        })

    return formatted_results

# ===============================
# 3️⃣ Display results
# ===============================
def display_results(results):
    if not results:
        print("❌ No relevant videos found.")
        return

    print("\n🎯 Top 5 Most Relevant Videos:")
    for r in results:
        print(f"\nRank {r['rank']}")
        print(f"Title: {r['title']}")
        print(f"Similarity Score: {r['similarity_score']}")
        print(f"Transcript Preview: {r['transcript']}")

# ===============================
# 🚀 Run the search
# ===============================
if __name__ == "__main__":
    user_query = input("🔍 Enter your search query: ").strip()
    if not user_query:
        print("❌ Please enter a valid query.")
    else:
        top_results = search_top_videos(user_query, top_k=5)
        display_results(top_results)



🎯 Top 5 Most Relevant Videos:

Rank 1
Title: why learning python won t land you a job in tech
Similarity Score: 0.529
Transcript Preview: please don't hate me, but if you want to land a job, you should probably stop learning python. when everyone knows python, it's no longer valuable. now, sure, it's good to know, but you're not going t...

Rank 2
Title: python is changing here s what s coming
Similarity Score: 0.518
Transcript Preview: just like programming is evolving, so is python. the way that we use python today is drastically different than five or 10 years ago. and in this video, i want to get into that in a lot more detail. n...

Rank 3
Title: how to become a python web dev in 2025
Similarity Score: 0.5
Transcript Preview: if you want to build a website in python, then you're in the right place. now, python offers some incredibly powerful yet beginnerfriendly frameworks that can get you from zero to a live website in li...

Rank 4
Title: how to go from 0 to 100 in python part 